# Correlogram PCA — Class-Based Pipeline Demo

Runs PCA across the Bragg-ring correlograms of every tile in a video, using the class-based rebuild of `Correlogram_PCA.ipynb`'s `Correlogram5DDataset`:

- **`TiledVideo`** — owns the video tensor + zero-copy tile grid (`twist_mapper_GPU/tiled_video.py`).
- **`TileFFTAnalysis`** — per-tile FFT magnitude, Bragg-ring-radius detection, angular intensity profile (`twist_mapper_GPU/tile_fft_analysis.py`).
- **`CorrelogramDataset`** — composes the two above: for every tile, refines the Bragg ring radius, computes the angular profile, then the 1D circular autocorrelation ("correlogram", Wiener-Khinchin) of that profile — a feature vector per tile suitable for PCA (`twist_mapper_GPU/correlogram_dataset.py`).

PCA runs on the GPU via `cuml.decomposition.PCA` (zero-copy from the `torch` tensors `CorrelogramDataset` returns) when RAPIDS is installed (Linux/WSL2 + NVIDIA), and on the CPU via `sklearn.decomposition.PCA` otherwise — same result either way. Set `N_COMPONENTS` below to control how many principal components are computed and plotted.

**Data:** runs on the sample image `data/moire.npz` by default — fetch it first with `python download_sample_data.py` (from this `examples/` folder; see `data/README.md` for the source and citation). To analyze your own data, set `SOURCE_PATH` below to any video, image, or `.npz` file.

In [ ]:
import time
import torch
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt

# GPU PCA via RAPIDS when available (Linux/WSL2 + NVIDIA), else scikit-learn on the CPU
try:
    import cupy as cp
    from cuml.decomposition import PCA
    HAS_RAPIDS = True
    to_numpy = cp.asnumpy
except ImportError:
    from sklearn.decomposition import PCA
    HAS_RAPIDS = False
    to_numpy = np.asarray

from twist_mapper_GPU import (TiledVideo, TileFFTAnalysis, CorrelogramDataset, load_video_tensor,
                              extract_pixel_size_ang, default_d_spacing_ang, get_material)

print(f"CUDA available: {torch.cuda.is_available()} | PCA backend: {'cuML (GPU)' if HAS_RAPIDS else 'scikit-learn (CPU)'}")

## Build the tile grid

`crop_black_margins=True` strips black padding (e.g. a video's pillarbox bars) before tiling (see `twist_mapper_GPU/tiled_video.py`), so no tile lands on dead padding.

In [ ]:
import os
SOURCE_PATH = os.path.join('data', 'moire.npz')  # or your own video / image / .npz
TILE_SIZE = 128
STRIDE = 32
PIXEL_SIZE_ANG = extract_pixel_size_ang(SOURCE_PATH) or 0.25  # from file metadata (.npz), else manual
MATERIAL = 'WSSe'
D_SPACING_ANG = default_d_spacing_ang(get_material(MATERIAL))
SUBTRACT_MEAN = False  # TileFFTAnalysis/CorrelogramDataset: subtract each tile's
                       # mean before windowing/FFT (reduces DC; does not sharpen
                       # Bragg peaks — see TileFFTAnalysis._compute_fft_magnitude)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tiled = TiledVideo(load_video_tensor(SOURCE_PATH, device), tile_size=TILE_SIZE, stride=STRIDE,
                   crop_black_margins=True)
print(tiled)

expected_radius = TileFFTAnalysis.expected_ring_radius(
    tiled.tile_size, pixel_size_ang=PIXEL_SIZE_ANG, d_spacing_ang=D_SPACING_ANG
)
print(f"expected_radius = {expected_radius:.2f} px")

## Build the correlogram dataset

`CorrelogramDataset` covers every `(frame, row, col)` tile in `tiled`'s grid. The full video has `Frames x Rows x Cols` tiles, which can be large — `FRAME_SAMPLE_STEP` subsamples frames (every Nth frame, all rows/cols) to keep this demo's runtime reasonable while still covering the full time range. Set it to `1` to use every frame.

In [ ]:
FRAME_SAMPLE_STEP = 30  # every 30th frame; set to 1 to use every frame

dataset = CorrelogramDataset(tiled, expected_radius=expected_radius, tolerance=0.15, n_bins=360,
                             subtract_mean=SUBTRACT_MEAN)
print(dataset)

n_frames = tiled.view_5d.shape[0]
sample_frames = list(range(0, n_frames, FRAME_SAMPLE_STEP))
flat_indices = [
    f * tiled.n_rows * tiled.n_cols + r * tiled.n_cols + c
    for f in sample_frames
    for r in range(tiled.n_rows)
    for c in range(tiled.n_cols)
]
subset = Subset(dataset, flat_indices)
print(f"Sampling {len(sample_frames)} of {n_frames} frames -> {len(subset)} tiles")

## Stream correlograms into one matrix

`num_workers=0` because every item's compute already happens on the GPU (see `CorrelogramDataset`'s docstring for the trade-off this implies).

In [ ]:
BATCH_SIZE = 1024

dataloader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

correlogram_blocks = []
coord_blocks = []

t0 = time.time()
for batch_correlograms, batch_coords in dataloader:
    correlogram_blocks.append(batch_correlograms)  # stays on the GPU if there is one
    coord_blocks.append(batch_coords.cpu().numpy())

X = torch.cat(correlogram_blocks)
spatial_coords = np.vstack(coord_blocks)
print(f"Collected X {tuple(X.shape)} on {X.device} in {time.time() - t0:.1f}s")

# Global standardization across the dataset for well-conditioned PCA
X = (X - X.mean(dim=0)) / (X.std(dim=0, correction=0) + 1e-8)

## Run PCA

`N_COMPONENTS` controls how many principal components are fit and plotted below.

In [ ]:
N_COMPONENTS = 4  # <-- set the number of PCs here

pca = PCA(n_components=N_COMPONENTS)
# torch CUDA tensor -> cupy array is zero-copy; scikit-learn needs a NumPy array on the CPU
X_pca = pca.fit_transform(cp.asarray(X) if HAS_RAPIDS else X.cpu().numpy())

explained = to_numpy(pca.explained_variance_ratio_)
print(f"X_pca shape: {X_pca.shape}")
for i, ev in enumerate(explained):
    print(f"  PC{i + 1}: {ev * 100:.2f}% variance explained (cumulative {explained[:i + 1].sum() * 100:.2f}%)")

## Plot all PCs

Each panel is one principal component's loading curve over the correlogram's angular-lag axis (`[-180, 180)` degrees) — the pattern of angular-displacement correlation structure that component captures.

In [ ]:
def plot_pca_components(pca_model, n_bins: int, explained_variance_ratio=None, ncols: int = 2):
    """Plots every fitted PC's loading curve over the correlogram lag axis."""
    components = to_numpy(pca_model.components_)  # (n_components, n_bins)
    n_components = components.shape[0]
    lags = np.linspace(-180, 180, n_bins, endpoint=False)

    ncols = min(ncols, n_components)
    nrows = int(np.ceil(n_components / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 3 * nrows), squeeze=False)

    for i in range(n_components):
        ax = axes[i // ncols][i % ncols]
        ax.plot(lags, components[i], lw=1.2, color='#1f77b4')
        ax.axvline(0, color='gray', ls=':', lw=1)
        title = f'PC{i + 1}'
        if explained_variance_ratio is not None:
            title += f' ({explained_variance_ratio[i] * 100:.1f}% var)'
        ax.set_title(title)
        ax.set_xlabel('Angular lag (degrees)')
        ax.set_ylabel('Loading')
        ax.grid(True, alpha=0.3)

    for j in range(n_components, nrows * ncols):
        axes[j // ncols][j % ncols].axis('off')

    fig.suptitle(f'Correlogram PCA — {n_components} components', fontsize=13, weight='bold')
    plt.tight_layout()
    plt.show()


plot_pca_components(pca, n_bins=dataset.n_bins, explained_variance_ratio=explained)